In [1]:
import sqlite3
from pathlib import Path

db = Path(
    "template_models/"
    "Soy_Template_v2.0_APSIM2026.db"
)

assert db.exists(), f"Database not found: {db}"

with sqlite3.connect(db) as conn:
    n = conn.execute(
        "SELECT COUNT(*) FROM Report"
    ).fetchone()[0]

    columns = [
        row[1] for row in
        conn.execute("PRAGMA table_info(Report)")
    ]

print("Report rows:", n)
print("Report columns:", columns)

assert n > 0, "APSIM Report is empty"

Report rows: 365
Report columns: ['CheckpointID', 'SimulationID', 'Zone', 'Date', 'Year', 'DOY', 'StageName', 'Stage', 'SimSowDate', 'SimMatDate', 'SimHarvestDate', 'Rain', 'Radn', 'MaxT', 'MeanT', 'MinT', 'ThermalTime', 'AccEmTT', 'SoilTemp', 'PAWmm', 'NH4', 'NO3', 'Urea', 'OrganicC', 'FracGrowthRate', 'NutrientStress', 'TempStress', 'WaterStress', 'Result']


In [2]:
import json
from pathlib import Path

original = Path("template_models/Soy_Template_v2.0_APSIM2026.apsimx")
copied = Path("tests/cli_smoketest/Soy_Template_v2.0_APSIM2026.apsimx")

for path in [original, copied]:
    model = json.loads(path.read_text())

    def find_weather(node):
        if isinstance(node, dict):
            if "Models.Climate.Weather" in node.get("$type", ""):
                return node.get("FileName")
            for child in node.get("Children", []):
                result = find_weather(child)
                if result is not None:
                    return result
        return None

    filename = find_weather(model)
    weather = Path(filename).expanduser()

    if not weather.is_absolute():
        weather = path.parent / weather

    print(f"\nModel: {path}")
    print(f"Weather reference: {filename}")
    print(f"Resolved path: {weather.resolve()}")
    print(f"File exists: {weather.exists()}")


Model: template_models/Soy_Template_v2.0_APSIM2026.apsimx
Weather reference: z_12.met
Resolved path: /Users/elhan/PycharmProjects/DigitalFields/external/SCE-main/template_models/z_12.met
File exists: True

Model: tests/cli_smoketest/Soy_Template_v2.0_APSIM2026.apsimx
Weather reference: z_12.met
Resolved path: /Users/elhan/PycharmProjects/DigitalFields/external/SCE-main/tests/cli_smoketest/z_12.met
File exists: False


In [3]:
import json
from pathlib import Path

original = Path("template_models/Soy_Template_v2.0_APSIM2026.apsimx")
copied = Path("tests/cli_smoketest/Soy_Template_v2.0_APSIM2026.apsimx")

model = json.loads(copied.read_text())

def fix_weather(node):
    if isinstance(node, dict):
        if "Models.Climate.Weather" in node.get("$type", ""):
            filename = Path(node["FileName"]).expanduser()
            if not filename.is_absolute():
                filename = (original.parent / filename).resolve()
            if not filename.is_file():
                raise FileNotFoundError(filename)
            node["FileName"] = str(filename)

        for child in node.get("Children", []):
            fix_weather(child)

fix_weather(model)
copied.write_text(json.dumps(model, indent=2))

print("Updated copied model with an absolute weather-file path.")

Updated copied model with an absolute weather-file path.


In [1]:
import pandas as pd

stages = pd.read_csv(
    "phase4_output_2001_mid000/stage_summary.csv"
)

columns = [
    "stage_segment_id",
    "stage_name",
    "start_date",
    "end_date",
    "days",
    "ThermalTime_sum",
    "Rain_sum",
    "Radn_sum",
    "PAWmm_mean",
    "WaterStress_mean",
    "TempStress_mean",
    "NutrientStress_mean",
    "longest_consecutive_dry_days",
]

display(stages[columns].round(2))

,stage_segment_id,stage_name,start_date,end_date,days,ThermalTime_sum,Rain_sum,Radn_sum,PAWmm_mean,WaterStress_mean,TempStress_mean,NutrientStress_mean,longest_consecutive_dry_days
0,1,Germinating,2001-04-12,2001-04-12,1,5.23,7.0,16.57,349.64,1.00,0.48,1.00,0
1,2,Emerging,2001-04-13,2001-04-29,17,39.66,38.0,289.08,351.44,1.00,0.17,1.00,5
2,3,Vegetative,2001-04-30,2001-06-08,40,302.25,27.0,942.15,327.96,1.00,0.66,0.94,23
3,4,EarlyFlowering,2001-06-09,2001-06-17,9,102.14,2.0,214.96,282.90,1.00,0.99,0.98,7
4,5,EarlyPodDevelopment,2001-06-18,2001-07-02,15,143.53,7.0,363.58,224.51,0.84,0.85,1.00,7
5,6,EarlyGrainFilling,2001-07-03,2001-07-23,21,251.87,127.0,359.55,220.99,0.96,1.00,1.00,3
6,7,MidGrainFilling,2001-07-24,2001-08-05,13,140.84,78.0,252.33,230.92,1.00,0.97,1.00,5
7,8,LateGrainFilling,2001-08-06,2001-08-18,13,133.52,0.0,281.55,186.25,0.94,0.85,1.00,13
8,9,Maturing,2001-08-19,2001-08-23,5,47.85,0.0,111.75,153.30,0.93,0.88,1.00,5
9,10,Ripening,2001-08-24,2001-08-26,3,41.19,0.0,56.03,147.58,1.00,1.00,1.00,3


In [2]:
import pandas as pd

stages = pd.read_csv(
    "phase4_output_2001_mid000_v2/stage_summary.csv"
)

columns = [
    "stage_name",
    "stage_class",
    "days",
    "WaterStress_mean",
    "WaterStress_deficit_factor_days",
    "WaterStress_days_below_threshold",
    "WaterStress_max_consecutive_days_below_threshold",
    "WaterStress_stress_observed_days",
]

print(stages[columns].to_string(index=False))

         stage_name        stage_class  days  WaterStress_mean  WaterStress_deficit_factor_days  WaterStress_days_below_threshold  WaterStress_max_consecutive_days_below_threshold  WaterStress_stress_observed_days
        Germinating       phenological     1          1.000000                         0.000000                                 0                                                 0                                 1
           Emerging       phenological    17          1.000000                         0.000000                                 0                                                 0                                17
         Vegetative       phenological    40          1.000000                         0.000000                                 0                                                 0                                40
     EarlyFlowering       phenological     9          1.000000                         0.000000                                 0               

In [3]:
print(
    stages[columns].to_string(
        index=False,
        max_rows=None,
        max_cols=None
    )
)

         stage_name        stage_class  days  WaterStress_mean  WaterStress_deficit_factor_days  WaterStress_days_below_threshold  WaterStress_max_consecutive_days_below_threshold  WaterStress_stress_observed_days
        Germinating       phenological     1          1.000000                         0.000000                                 0                                                 0                                 1
           Emerging       phenological    17          1.000000                         0.000000                                 0                                                 0                                17
         Vegetative       phenological    40          1.000000                         0.000000                                 0                                                 0                                40
     EarlyFlowering       phenological     9          1.000000                         0.000000                                 0               

In [4]:
import pandas as pd

summary = pd.read_csv(
    "phase5_output_2001_mid000/run_summary.csv"
)

comparison = pd.read_csv(
    "phase5_output_2001_mid000/stage_comparison.csv"
)

display(summary)
display(comparison)

,run_id,site,year,cultivar,maturity_group,sowing_date,harvest_date,season_days,biological_segments,harvest_transition_days,rain_season_mm,thermal_time_season
0,IL_2001_mid000,Illinois example,2001,mid000,0,2001-04-12,2001-08-28,139,11,1,307.0,1215.564205


,run_id,site,year,cultivar,maturity_group,stage_segment_id,stage_name,start_date,end_date,days,Rain_sum,Radn_sum,ThermalTime_sum,WaterStress_mean,WaterStress_deficit_factor_days,WaterStress_days_below_threshold,WaterStress_max_consecutive_days_below_threshold
0,IL_2001_mid000,Illinois example,2001,mid000,0,1,Germinating,2001-04-12,2001-04-12,1,7.0,16.572212,5.229331,1.000000,0.000000,0,0
1,IL_2001_mid000,Illinois example,2001,mid000,0,2,Emerging,2001-04-13,2001-04-29,17,38.0,289.076432,39.655684,1.000000,0.000000,0,0
2,IL_2001_mid000,Illinois example,2001,mid000,0,3,Vegetative,2001-04-30,2001-06-08,40,27.0,942.145409,302.250982,1.000000,0.000000,0,0
3,IL_2001_mid000,Illinois example,2001,mid000,0,4,EarlyFlowering,2001-06-09,2001-06-17,9,2.0,214.957668,102.137759,1.000000,0.000000,0,0
4,IL_2001_mid000,Illinois example,2001,mid000,0,5,EarlyPodDevelopment,2001-06-18,2001-07-02,15,7.0,363.576733,143.530772,0.835816,2.462758,7,7
5,IL_2001_mid000,Illinois example,2001,mid000,0,6,EarlyGrainFilling,2001-07-03,2001-07-23,21,127.0,359.551178,251.868002,0.964715,0.740990,1,1
6,IL_2001_mid000,Illinois example,2001,mid000,0,7,MidGrainFilling,2001-07-24,2001-08-05,13,78.0,252.326709,140.836291,0.998395,0.020866,0,0
7,IL_2001_mid000,Illinois example,2001,mid000,0,8,LateGrainFilling,2001-08-06,2001-08-18,13,0.0,281.548429,133.519066,0.937527,0.812147,1,1
8,IL_2001_mid000,Illinois example,2001,mid000,0,9,Maturing,2001-08-19,2001-08-23,5,0.0,111.754322,47.850225,0.928932,0.355342,0,0
9,IL_2001_mid000,Illinois example,2001,mid000,0,10,Ripening,2001-08-24,2001-08-26,3,0.0,56.032545,41.186094,1.000000,0.000000,0,0


In [5]:
import pandas as pd

folder = "multiyear_2001_2005_2010/characterization"

summary = pd.read_csv(f"{folder}/run_summary.csv")
wide = pd.read_csv(f"{folder}/environment_matrix_wide.csv")

display(summary)
print("Wide matrix shape:", wide.shape)
display(wide[["run_id", "year", "cultivar"]])

,run_id,site,year,cultivar,maturity_group,sowing_date,harvest_date,season_days,biological_segments,harvest_transition_days,rain_season_mm,thermal_time_season
0,IL_2001_mid000,Illinois example,2001,mid000,0,2001-04-12,2001-08-28,139,11,1,307.0,1215.564205
1,IL_2005_mid000,Illinois example,2005,mid000,0,2005-04-12,2005-08-12,123,11,1,375.0,1216.702806
2,IL_2010_mid000,Illinois example,2010,mid000,0,2010-04-12,2010-08-27,138,11,1,537.0,1226.648420


Wide matrix shape: (3, 599)


,run_id,year,cultivar
0,IL_2001_mid000,2001,mid000
1,IL_2005_mid000,2005,mid000
2,IL_2010_mid000,2010,mid000


In [6]:
import pandas as pd

stages = pd.read_csv(
    "multiyear_2001_2005_2010/characterization/batch_stages.csv"
)

cols = [
    "year",
    "stage_name",
    "days",
    "Rain_sum",
    "WaterStress_mean",
    "WaterStress_deficit_factor_days",
    "TempStress_mean"
]

comparison = stages[
    (stages["stage_class"] == "phenological") &
    (stages["stage_name"].isin([
        "EarlyFlowering",
        "EarlyPodDevelopment",
        "EarlyGrainFilling",
        "MidGrainFilling",
        "LateGrainFilling"
    ]))
][cols]

print(comparison.round(3).to_string(index=False))

 year          stage_name  days  Rain_sum  WaterStress_mean  WaterStress_deficit_factor_days  TempStress_mean
 2001      EarlyFlowering     9       2.0             1.000                            0.000            0.989
 2001 EarlyPodDevelopment    15       7.0             0.836                            2.463            0.852
 2001   EarlyGrainFilling    21     127.0             0.965                            0.741            0.995
 2001     MidGrainFilling    13      78.0             0.998                            0.021            0.967
 2001    LateGrainFilling    13       0.0             0.938                            0.812            0.848
 2005      EarlyFlowering    10      44.0             0.965                            0.349            0.830
 2005 EarlyPodDevelopment    13      40.0             0.989                            0.143            0.942
 2005   EarlyGrainFilling    20      24.0             0.918                            1.639            0.988
 2005     

In [7]:
import pandas as pd
from pathlib import Path

folder = Path("multiyear_2001_2005_2010/characterization")

wide = pd.read_csv(folder / "environment_matrix_wide.csv")
stages = pd.read_csv(folder / "batch_stages.csv")

print("Matrix shape:", wide.shape)
print("Simulations:", wide["run_id"].tolist())
print("Duplicate column names:", wide.columns.duplicated().sum())
print("Missing values:", wide.isna().sum().sum())

display(wide.iloc[:, :12])

Matrix shape: (3, 599)
Simulations: ['IL_2001_mid000', 'IL_2005_mid000', 'IL_2010_mid000']
Duplicate column names: 0
Missing values: 0


,run_id,site,year,cultivar,maturity_group,EarlyFlowering__occurrence01__AccEmTT_end,EarlyFlowering__occurrence01__AccEmTT_start,EarlyFlowering__occurrence01__FracGrowthRate_mean,EarlyFlowering__occurrence01__FracGrowthRate_observed_days,EarlyFlowering__occurrence01__MaxT_mean,EarlyFlowering__occurrence01__MaxT_observed_days,EarlyFlowering__occurrence01__MeanT_mean
0,IL_2001_mid000,Illinois example,2001,mid000,0,404.388741,310.935712,0.977201,9,29.166667,9,21.972222
1,IL_2005_mid000,Illinois example,2005,mid000,0,405.001663,310.457267,0.812425,10,26.100000,10,20.425000
2,IL_2010_mid000,Illinois example,2010,mid000,0,403.007111,321.975497,0.993315,7,29.571429,7,24.428571


In [8]:
stage = "EarlyPodDevelopment"
variable = "WaterStress_deficit_factor_days"

column = f"{stage}__occurrence01__{variable}"

assert column in wide.columns, f"Missing column: {column}"
assert wide["run_id"].is_unique, "Duplicate simulation IDs"
assert not wide.columns.duplicated().any(), "Duplicate column names"

original = stages.loc[
    (stages["stage_name"] == stage) &
    (stages["stage_class"] == "phenological"),
    ["run_id", variable]
]

comparison = wide[["run_id", column]].merge(
    original,
    on="run_id",
    validate="one_to_one"
)

comparison["difference"] = (
    comparison[column] - comparison[variable]
)

display(comparison)

assert comparison["difference"].abs().max() < 1e-8

print("PASS: Wide matrix matches original stage summaries.")

,run_id,EarlyPodDevelopment__occurrence01__WaterStress_deficit_factor_days,WaterStress_deficit_factor_days,difference
0,IL_2001_mid000,2.462758,2.462758,0.0
1,IL_2005_mid000,0.142725,0.142725,0.0
2,IL_2010_mid000,0.000000,0.000000,0.0


PASS: Wide matrix matches original stage summaries.


In [9]:
import numpy as np

id_cols = ["run_id", "year", "cultivar"]

assert len(wide) == 3
assert wide["run_id"].is_unique
assert not wide.columns.duplicated().any()

numeric_cols = [
    c for c in wide.columns
    if c not in id_cols
    and pd.api.types.is_numeric_dtype(wide[c])
]

invalid = [
    c for c in numeric_cols
    if np.isinf(wide[c].to_numpy(dtype=float)).any()
]

assert not invalid, f"Infinite values: {invalid}"

print("PASS: Three unique simulations.")
print("Numeric covariates:", len(numeric_cols))
print("Infinite-value columns:", len(invalid))
print("Missing-value count:", wide.isna().sum().sum())

PASS: Three unique simulations.
Numeric covariates: 595
Infinite-value columns: 0
Missing-value count: 0


In [10]:
import sqlite3
import pandas as pd

db = "rotation_native_smoketest/IL_2001_mid000_drill.db"

with sqlite3.connect(db) as con:
    df = pd.read_sql_query('SELECT * FROM "Report"', con)

print("Report columns:", df.columns.tolist())

if "SowingDate" in df.columns:
    print("Reported sowing dates:", df["SowingDate"].dropna().unique())

if "Phenology" in df.columns:
    print("Phenological stages:")
    print(df["Phenology"].dropna().value_counts())

print("First five report records:")
print(df.head())

Report columns: ['CheckpointID', 'SimulationID', 'Zone', 'Date', 'Year', 'DOY', 'StageName', 'Stage', 'SimSowDate', 'SimMatDate', 'SimHarvestDate', 'Rain', 'Radn', 'MaxT', 'MeanT', 'MinT', 'ThermalTime', 'AccEmTT', 'SoilTemp', 'PAWmm', 'NH4', 'NO3', 'Urea', 'OrganicC', 'FracGrowthRate', 'NutrientStress', 'TempStress', 'WaterStress', 'Result']
First five report records:
   CheckpointID  SimulationID   Zone                 Date  Year  DOY  \
0             1             1  Field  2001-01-01 00:00:00  2001    1   
1             1             1  Field  2001-01-02 00:00:00  2001    2   
2             1             1  Field  2001-01-03 00:00:00  2001    3   
3             1             1  Field  2001-01-04 00:00:00  2001    4   
4             1             1  Field  2001-01-05 00:00:00  2001    5   

  StageName  Stage SimSowDate SimMatDate  ...       PAWmm       NH4  \
0              1.0         NA         NA  ...  360.879731  2.045116   
1              1.0         NA         NA  ...  362.44

In [11]:
import sqlite3
import pandas as pd

db = "rotation_native_smoketest/IL_2001_mid000_drill.db"

with sqlite3.connect(db) as con:
    df = pd.read_sql_query('SELECT * FROM "Report"', con)

for col in ["SimSowDate", "SimMatDate", "SimHarvestDate"]:
    values = df[col].astype(str)
    dates = values[~values.isin(["NA", "", "None", "nan"])]
    print(f"{col}: {dates.unique()}")

print("\nFirst biological stages:")
print(
    df.loc[
        df["StageName"].notna() &
        (df["StageName"].astype(str).str.strip() != ""),
        ["Date", "StageName", "Stage"]
    ].head(15).to_string(index=False)
)

print("\nFinal biological stages:")
print(
    df.loc[
        df["StageName"].notna() &
        (df["StageName"].astype(str).str.strip() != ""),
        ["Date", "StageName", "Stage"]
    ].tail(15).to_string(index=False)
)


SimSowDate: <StringArray>
['2001-04-12']
Length: 1, dtype: str
SimMatDate: <StringArray>
['2001-08-25']
Length: 1, dtype: str
SimHarvestDate: <StringArray>
['2001-08-28']
Length: 1, dtype: str

First biological stages:
               Date   StageName    Stage
2001-04-12 00:00:00 Germinating 1.000000
2001-04-13 00:00:00    Emerging 2.000720
2001-04-14 00:00:00    Emerging 2.000720
2001-04-15 00:00:00    Emerging 2.034915
2001-04-16 00:00:00    Emerging 2.174369
2001-04-17 00:00:00    Emerging 2.339259
2001-04-18 00:00:00    Emerging 2.385600
2001-04-19 00:00:00    Emerging 2.460327
2001-04-20 00:00:00    Emerging 2.641577
2001-04-21 00:00:00    Emerging 2.810327
2001-04-22 00:00:00    Emerging 2.867163
2001-04-23 00:00:00    Emerging 2.872798
2001-04-24 00:00:00    Emerging 2.918313
2001-04-25 00:00:00    Emerging 2.952155
2001-04-26 00:00:00    Emerging 2.952155

Final biological stages:
               Date          StageName     Stage
2001-08-13 00:00:00   LateGrainFilling  8.666173
2

In [12]:
import sqlite3
import pandas as pd

databases = {
    "Original": "rotation_native_smoketest/IL_2001_mid000_drill.db",
    "ERA5": "rotation_era5_2001/IL_2001_mid000_drill.db",
}

results = []

for source, database in databases.items():
    with sqlite3.connect(database) as con:
        df = pd.read_sql_query('SELECT * FROM "Report"', con)

    growing = df[
        (df["Date"] >= "2001-04-12") &
        (df["Date"] <= "2001-08-31")
    ].copy()

    def reported_date(column):
        values = df[column].astype(str)
        values = values[~values.isin(["NA", "", "None", "nan"])]
        return values.iloc[0] if len(values) else None

    results.append({
        "Weather": source,
        "Sowing": reported_date("SimSowDate"),
        "Maturity": reported_date("SimMatDate"),
        "Harvest": reported_date("SimHarvestDate"),
        "Rain_mm": growing["Rain"].sum(),
        "ThermalTime": growing["ThermalTime"].sum(),
        "WaterStress_mean": growing["WaterStress"].mean(),
        "TempStress_mean": growing["TempStress"].mean(),
    })

comparison = pd.DataFrame(results)
print(comparison.to_string(index=False))

 Weather     Sowing   Maturity    Harvest    Rain_mm  ThermalTime  WaterStress_mean  TempStress_mean
Original 2001-04-12 2001-08-25 2001-08-28 307.000000  1227.488387          0.964576         0.755496
    ERA5 2001-04-12 2001-08-05 2001-08-08 465.700001  1497.920444          0.977646         0.835567


In [13]:

import sqlite3
import pandas as pd

databases = {
    "Original": "rotation_native_smoketest/IL_2001_mid000_drill.db",
    "ERA5": "rotation_era5_2001/IL_2001_mid000_drill.db",
}

results = []

for source, database in databases.items():
    with sqlite3.connect(database) as con:
        df = pd.read_sql_query('SELECT * FROM "Report"', con)

    df["Date"] = pd.to_datetime(df["Date"])

    sowing = pd.to_datetime(
        df.loc[df["SimSowDate"] != "NA", "SimSowDate"].iloc[0]
    )
    harvest = pd.to_datetime(
        df.loc[df["SimHarvestDate"] != "NA", "SimHarvestDate"].iloc[0]
    )

    growing = df.loc[
        (df["Date"] >= sowing) &
        (df["Date"] <= harvest)
    ].copy()

    results.append({
        "Weather": source,
        "Growing_days": len(growing),
        "Rain_mm": growing["Rain"].sum(),
        "Radiation_MJ_m2": growing["Radn"].sum(),
        "MeanT_C": growing["MeanT"].mean(),
        "MaxT_C": growing["MaxT"].mean(),
        "MinT_C": growing["MinT"].mean(),
        "ThermalTime": growing["ThermalTime"].sum(),
        "WaterStress": growing["WaterStress"].mean(),
        "TempStress": growing["TempStress"].mean(),
    })

comparison = pd.DataFrame(results)
print(comparison.round(3).to_string(index=False))


 Weather  Growing_days  Rain_mm  Radiation_MJ_m2  MeanT_C  MaxT_C  MinT_C  ThermalTime  WaterStress  TempStress
Original           139    307.0         2910.193   18.451  24.899  12.004     1215.564        0.964       0.758
    ERA5           119    380.6         2626.900   20.600  25.534  15.666     1225.745        0.974       0.808


In [ ]:
python3 scripts/prepare_site.py \
  --interactive \
  --output-dir site_illinois_v1

In [14]:
import pandas as pd

path = "data/soil_cache/dfc3c56f88cdddcc/soil_layers.csv"

soil = pd.read_csv(path)

columns = [
    "top_cm",
    "bottom_cm",
    "sand_pct",
    "silt_pct",
    "clay_pct",
    "bulk_density_g_cm3",
    "water_33kpa_pct",
    "water_1500kpa_pct",
    "organic_matter_pct",
    "ph",
    "ksat_um_s",
]

print(soil[columns].to_string(index=False))
print("\nMissing values:")
print(soil[columns].isna().sum())

 top_cm  bottom_cm  sand_pct  silt_pct  clay_pct  bulk_density_g_cm3  water_33kpa_pct  water_1500kpa_pct  organic_matter_pct  ph  ksat_um_s
      0          8         4        77        19                1.45             28.3               13.0                1.50 6.8       9.17
      8         71         4        65        31                1.45             31.9               18.3                0.25 5.5       9.17
     71        100         4        72        24                1.45             29.2               14.2                0.25 6.0       9.17
    100        152         4        72        24                1.35             28.5               13.3                0.25 6.2       9.17

Missing values:
top_cm                0
bottom_cm             0
sand_pct              0
silt_pct              0
clay_pct              0
bulk_density_g_cm3    0
water_33kpa_pct       0
water_1500kpa_pct     0
organic_matter_pct    0
ph                    0
ksat_um_s             0
dtype: int64


In [15]:

import json
from pathlib import Path

template = Path(
    "template_models/Soy_Template_v2.0_APSIM2026.apsimx"
)

model = json.loads(template.read_text())

def walk(obj):
    if isinstance(obj, dict):
        yield obj
        for value in obj.values():
            yield from walk(value)
    elif isinstance(obj, list):
        for value in obj:
            yield from walk(value)

physical = next(
    n for n in walk(model)
    if n.get("$type", "").split(",")[0].strip()
    == "Models.Soils.Physical"
)

soybean = next(
    n for n in walk(physical)
    if n.get("Name") == "SoybeanSoil"
)

print("Layer | Thickness | Existing LL | Existing DUL | New Fayette DUL")
print("-" * 73)

# Proposed Fayette DUL must be remapped to the APSIM layers.
# First inspect the original arrays before updating them.
for i, (thickness, ll, dul) in enumerate(
    zip(
        physical["Thickness"],
        soybean["LL"],
        physical["DUL"],
    ),
    start=1,
):
    print(
        f"{i:5} | {thickness:9} | "
        f"{ll:11.4f} | {dul:12.4f}"
    )

print("\nSoybean KL:", soybean.get("KL"))
print("Soybean XF:", soybean.get("XF"))
print("Physical LL15:", physical.get("LL15"))
print("Physical AirDry:", physical.get("AirDry"))


Layer | Thickness | Existing LL | Existing DUL | New Fayette DUL
-------------------------------------------------------------------------
    1 |     150.0 |      0.2610 |       0.5210
    2 |     150.0 |      0.2480 |       0.4970
    3 |     300.0 |      0.2800 |       0.4880
    4 |     300.0 |      0.3060 |       0.4800
    5 |     300.0 |      0.3600 |       0.4720
    6 |     300.0 |      0.3920 |       0.4570
    7 |     300.0 |      0.4460 |       0.4520

Soybean KL: [0.06, 0.06, 0.06, 0.04, 0.04, 0.02, 0.01]
Soybean XF: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Physical LL15: [0.261, 0.248, 0.28, 0.28, 0.28, 0.28, 0.28]
Physical AirDry: [0.13, 0.199, 0.28, 0.28, 0.28, 0.28, 0.28]


In [16]:

import json
from pathlib import Path

path = Path("soil_fayette_test/Fayette.apsimx")
model = json.loads(path.read_text())

def walk(obj):
    if isinstance(obj, dict):
        yield obj
        for value in obj.values():
            yield from walk(value)
    elif isinstance(obj, list):
        for value in obj:
            yield from walk(value)

for node in walk(model):
    if node.get("$type", "").split(",")[0].strip() in (
        "Models.Soils.Physical",
        "Models.Soils.Water",
        "Models.Soils.SoilWater",
        "Models.Soils.InitialWater",
    ):
        print("\nNODE:", node.get("Name"))
        print("TYPE:", node.get("$type"))
        for key in (
            "Thickness", "InitialValues", "InitialWater",
            "SW", "AirDry", "LL15", "DUL", "SAT"
        ):
            if key in node:
                print(key, "=", node[key])



NODE: Physical
TYPE: Models.Soils.Physical, Models
Thickness = [150.0, 150.0, 300.0, 300.0, 300.0, 300.0, 300.0]
AirDry = [0.13, 0.199, 0.28, 0.28, 0.28, 0.28, 0.28]
LL15 = [0.15473333333333333, 0.183, 0.183, 0.15703333333333333, 0.136, 0.133, 0.133]
DUL = [0.2998, 0.319, 0.319, 0.3019, 0.28733333333333333, 0.285, 0.285]
SAT = [0.4528301886792453, 0.4528301886792453, 0.4528301886792453, 0.4528301886792453, 0.4779874213836478, 0.49056603773584906, 0.49056603773584906]

NODE: Water
TYPE: Models.Soils.Water, Models
Thickness = [150.0, 150.0, 300.0, 300.0, 300.0, 300.0, 300.0]
InitialValues = [0.521, 0.497, 0.488, 0.48, 0.472, 0.457, 0.452]


In [17]:
import json
from pathlib import Path

model = json.loads(
    Path("soil_fayette_test/Fayette_v2.apsimx").read_text()
)

def walk(obj):
    if isinstance(obj, dict):
        yield obj
        for value in obj.values():
            yield from walk(value)
    elif isinstance(obj, list):
        for value in obj:
            yield from walk(value)

physical = next(
    n for n in walk(model)
    if n.get("$type", "").split(",")[0].strip()
    == "Models.Soils.Physical"
)

print("Layer | AirDry | LL15 | DUL | SAT")
for i, values in enumerate(zip(
    physical["AirDry"],
    physical["LL15"],
    physical["DUL"],
    physical["SAT"]
), 1):
    air, ll, dul, sat = values
    print(f"{i:5} | {air:.3f} | {ll:.3f} | {dul:.3f} | {sat:.3f}")
    assert 0 <= air <= ll <= dul < sat

print("\nPASS: Hydraulic limits are internally consistent.")

for node in walk(model):
    if node.get("$type", "").split(",")[0].strip() == "Models.Soils.Water":
        print("\nWater initialization:")
        print(json.dumps(
            {k: v for k, v in node.items() if k != "Children"},
            indent=2
        ))

Layer | AirDry | LL15 | DUL | SAT
    1 | 0.155 | 0.155 | 0.300 | 0.453
    2 | 0.183 | 0.183 | 0.319 | 0.453
    3 | 0.183 | 0.183 | 0.319 | 0.453
    4 | 0.157 | 0.157 | 0.302 | 0.453
    5 | 0.136 | 0.136 | 0.287 | 0.478
    6 | 0.133 | 0.133 | 0.285 | 0.491
    7 | 0.133 | 0.133 | 0.285 | 0.491

PASS: Hydraulic limits are internally consistent.

Water initialization:
{
  "$type": "Models.Soils.Water, Models",
  "Thickness": [
    150.0,
    150.0,
    300.0,
    300.0,
    300.0,
    300.0,
    300.0
  ],
  "InitialValues": [
    0.521,
    0.497,
    0.488,
    0.48,
    0.472,
    0.457,
    0.452
  ],
  "InitialPAWmm": 361.0499999999999,
  "RelativeTo": "LL15",
  "FilledFromTop": false,
  "Name": "Water",
  "ResourceName": null,
  "Enabled": true,
  "ReadOnly": false
}


In [18]:
import json
from pathlib import Path

path = Path("soil_fayette_test/Fayette_v2.apsimx")
model = json.loads(path.read_text())

def walk(obj):
    if isinstance(obj, dict):
        yield obj
        for value in obj.values():
            yield from walk(value)
    elif isinstance(obj, list):
        for value in obj:
            yield from walk(value)

physical = next(
    n for n in walk(model)
    if n.get("$type", "").split(",")[0].strip()
    == "Models.Soils.Physical"
)

water = next(
    n for n in walk(model)
    if n.get("$type", "").split(",")[0].strip()
    == "Models.Soils.Water"
)

capacity = sum(
    thickness * (dul - ll)
    for thickness, dul, ll in zip(
        physical["Thickness"],
        physical["DUL"],
        physical["LL15"]
    )
)

print(f"Fayette profile PAWC: {capacity:.2f} mm")

water["InitialPAWmm"] = capacity
water["RelativeTo"] = "LL15"
water["FilledFromTop"] = False

output = Path("soil_fayette_test/Fayette_v3.apsimx")
if output.exists():
    raise FileExistsError(output)

output.write_text(json.dumps(model, indent=2) + "\n")

print("Saved:", output)

Fayette profile PAWC: 263.02 mm
Saved: soil_fayette_test/Fayette_v3.apsimx


In [20]:

import json
import sqlite3
import subprocess
from pathlib import Path

model_path = Path(
    "soil_fayette_test/native_era5/Fayette_ERA5.apsimx"
).resolve()

weather_path = Path(
    "openmeteo_era5_illinois_2001_2010/weather.met"
).resolve()

apsim = Path(
    "/Applications/APSIM2026.10.8138.0.app/"
    "Contents/Resources/bin/Models"
)

model = json.loads(model_path.read_text())

def walk(obj):
    if isinstance(obj, dict):
        yield obj
        for value in obj.values():
            yield from walk(value)
    elif isinstance(obj, list):
        for value in obj:
            yield from walk(value)

def find(name):
    matches = [
        n for n in walk(model)
        if n.get("Name") == name
    ]
    if len(matches) != 1:
        raise ValueError(f"{name}: {len(matches)} matches")
    return matches[0]

weather_nodes = [
    n for n in walk(model)
    if "Models.Climate.Weather" in n.get("$type", "")
]

assert weather_nodes
for node in weather_nodes:
    node["FileName"] = str(weather_path)

clock = find("Clock")
clock["Start"] = "2001-01-01"
clock["End"] = "2001-12-31"

sowing = find("Sowing")
settings = {
    "SowDate": "12-apr",
    "CultivarName": "mid000",
    "Population": "40",
    "RowSpacing": "190",
    "SowingDepth": "30",
}
for parameter in sowing["Parameters"]:
    if parameter["Key"] in settings:
        parameter["Value"] = settings[parameter["Key"]]

find("Fertilise at sowing")["Enabled"] = False
find("Tillage on a fixed date")["Enabled"] = False

model_path.write_text(json.dumps(model, indent=2) + "\n")

result = subprocess.run(
    [str(apsim), str(model_path)],
    cwd=model_path.parent,
    capture_output=True,
    text=True
)

(model_path.parent / "apsim_stdout.log").write_text(result.stdout)
(model_path.parent / "apsim_stderr.log").write_text(result.stderr)

print("APSIM exit code:", result.returncode)

if result.returncode != 0:
    print(result.stderr[-4000:])
    raise RuntimeError("APSIM simulation failed")

database = model_path.with_suffix(".db")

with sqlite3.connect(database) as connection:
    count = connection.execute(
        'SELECT COUNT(*) FROM "Report"'
    ).fetchone()[0]

print("Daily report rows:", count)
print("Database:", database)


APSIM exit code: 1



RuntimeError: APSIM simulation failed

In [21]:
from pathlib import Path

folder = Path("soil_fayette_test/native_era5")

for filename in ["apsim_stderr.log", "apsim_stdout.log"]:
    path = folder / filename
    print(f"\n{'=' * 20} {filename} {'=' * 20}")

    if path.exists():
        text = path.read_text(errors="replace")
        print(text[-12000:] if text.strip() else "(empty)")
    else:
        print("File not found")


==================== apsim_stderr.log ====================
(empty)

==================== apsim_stdout.log ====================
----------------------------------------------
ERROR in file: /Users/elhan/PycharmProjects/DigitalFields/external/SCE-main/soil_fayette_test/native_era5/Fayette_ERA5.apsimx
Simulation name: Simulation
Models.Core.SimulationException
 ---> System.Exception: Soil water of 0.521 in layer 1 is above saturation of 0.453
Soil water of 0.497 in layer 2 is above saturation of 0.453
Soil water of 0.488 in layer 3 is above saturation of 0.453
Soil water of 0.480 in layer 4 is above saturation of 0.453

   at Models.Soils.Soil.Check(ISummary summary) in /__w/ApsimX/ApsimX/Models/Soils/Soil.cs:line 344
   at Models.Soils.Soil.OnDoInitialSummary(Object sender, EventArgs e) in /__w/ApsimX/ApsimX/Models/Soils/Soil.cs:line 132
   at Models.Clock.OnDoCommence(Object _, CommenceArgs e) in /__w/ApsimX/ApsimX/Models/Clock.cs:line 264
   at Models.Core.Simulation.Run(CancellationT

In [22]:

import json
from pathlib import Path

source = Path(
    "soil_fayette_test/native_era5/Fayette_ERA5.apsimx"
)
output = Path(
    "soil_fayette_test/native_era5/Fayette_ERA5_v2.apsimx"
)

if output.exists():
    raise FileExistsError(output)

model = json.loads(source.read_text())

def walk(obj):
    if isinstance(obj, dict):
        yield obj
        for value in obj.values():
            yield from walk(value)
    elif isinstance(obj, list):
        for value in obj:
            yield from walk(value)

physical = next(
    node for node in walk(model)
    if node.get("$type", "").split(",")[0].strip()
    == "Models.Soils.Physical"
)

water = next(
    node for node in walk(model)
    if node.get("$type", "").split(",")[0].strip()
    == "Models.Soils.Water"
)

dul = list(physical["DUL"])
sat = list(physical["SAT"])
ll15 = list(physical["LL15"])
thickness = list(physical["Thickness"])

assert all(
    0 <= ll < du < sa
    for ll, du, sa in zip(ll15, dul, sat)
)

print("Existing Water initialization fields:")
for key, value in water.items():
    if key != "Children":
        print(key, "=", value)

# This array contains the original soil water values.
if "InitialValues" not in water:
    raise ValueError(
        "Water.InitialValues not found. "
        "Inspect the printed fields before proceeding."
    )

if len(water["InitialValues"]) != len(dul):
    raise ValueError("Initial water layer count mismatch.")

# Set initial soil water to the new DUL.
water["InitialValues"] = dul

capacity = sum(
    t * (du - ll)
    for t, du, ll in zip(thickness, dul, ll15)
)

water["InitialPAWmm"] = capacity
water["RelativeTo"] = "LL15"
water["FilledFromTop"] = False

output.write_text(json.dumps(model, indent=2) + "\n")

print("\nCorrected initial soil water:")
for i, (sw, sa) in enumerate(
    zip(water["InitialValues"], sat), 1
):
    print(f"Layer {i}: SW={sw:.4f}, SAT={sa:.4f}")

print(f"\nInitial PAW: {capacity:.2f} mm")
print("Saved:", output)


Existing Water initialization fields:
$type = Models.Soils.Water, Models
Thickness = [150.0, 150.0, 300.0, 300.0, 300.0, 300.0, 300.0]
InitialValues = [0.521, 0.497, 0.488, 0.48, 0.472, 0.457, 0.452]
InitialPAWmm = 263.02
RelativeTo = LL15
FilledFromTop = False
Name = Water
ResourceName = None
Enabled = True
ReadOnly = False

Corrected initial soil water:
Layer 1: SW=0.2998, SAT=0.4528
Layer 2: SW=0.3190, SAT=0.4528
Layer 3: SW=0.3190, SAT=0.4528
Layer 4: SW=0.3019, SAT=0.4528
Layer 5: SW=0.2873, SAT=0.4780
Layer 6: SW=0.2850, SAT=0.4906
Layer 7: SW=0.2850, SAT=0.4906

Initial PAW: 263.02 mm
Saved: soil_fayette_test/native_era5/Fayette_ERA5_v2.apsimx


In [23]:
import sqlite3
from pathlib import Path

db = Path(
    "soil_fayette_test/native_era5/Fayette_ERA5_v2.db"
)

with sqlite3.connect(db) as con:
    count = con.execute(
        'SELECT COUNT(*) FROM "Report"'
    ).fetchone()[0]

    errors = con.execute(
        "SELECT name FROM sqlite_master WHERE type='table'"
    ).fetchall()

print("Daily report rows:", count)
print("Database tables:", errors)

Daily report rows: 365
Database tables: [('_InitialConditions',), ('Report',), ('_Messages',), ('_Simulations',), ('_Checkpoints',), ('_Units',)]
